# Bluesky Bot-Cluster Analyse

Diese Notebook erstellt einen vollständigen Bot-Cluster-Report rund um einen Bluesky-Anker-Account. Die einzige Konfiguration ist der `ANCHOR_HANDLE` in der Parameterzelle weiter unten.

**Voraussetzungen**
- Notebook ist mit einer Bluesky-KQL-Datenbank verbunden (KQL-Tab links). `deploy-fabric-notebook.ps1` setzt diese Bindung automatisch.
- Identität, mit der das Notebook ausgeführt wird, hat Reader-Rechte auf der Eventhouse-Datenbank.


In [ ]:
# Modul aus dem real-time-sources Repo installieren
%pip install -q git+https://github.com/clemensv/real-time-sources@main#subdirectory=bluesky/botfinder


In [ ]:
# === PARAMETERS ===
# Inhaltliche Variable: ANCHOR_HANDLE.
# KUSTO_URI / KUSTO_DATABASE werden vom Deploy-Skript pro Workspace gesetzt.
ANCHOR_HANDLE  = "niusde.bsky.social"
LOOKBACK_DAYS  = 90
MAX_FOLLOWERS  = 5000
ENRICH_LIMIT   = 400
SKIP_API       = False

# Workspace-spezifische Defaults — vom Deploy-Skript überschrieben.
KUSTO_URI      = ""
KUSTO_DATABASE = ""


In [ ]:
from botfinder import Config, run_full_pipeline

config = Config.from_fabric_context(
    anchor_handle=ANCHOR_HANDLE,
    kusto_uri=KUSTO_URI or None,
    kusto_database=KUSTO_DATABASE or None,
    lookback_days=LOOKBACK_DAYS,
    max_followers=MAX_FOLLOWERS,
    enrich_limit=ENRICH_LIMIT,
    skip_api=SKIP_API,
)
print(f"Cluster:  {config.kusto_uri}")
print(f"Database: {config.kusto_database}")
print(f"Anchor:   @{config.anchor_handle}")
assert config.kusto_uri and config.kusto_database, (
    "KQL database not configured. Re-deploy this notebook via "
    "deploy-fabric-notebook.ps1 or set KUSTO_URI / KUSTO_DATABASE "
    "in the parameters cell."
)


## Vollständige Pipeline ausführen
Akquise (KQL) → API-Anreicherung → Scoring → Cluster-Graph → Cross-Follow → Karten.

In [ ]:
result = run_full_pipeline(config)
scores = result.analysis.scores_df
print(f"Cohort: {len(scores)} accounts")
print(f"Bots:   {(scores['score'] >= config.bot_score_threshold).sum()}")


## KPI-Übersicht

In [ ]:
import pandas as pd
scores = result.analysis.scores_df
flags = scores['flags'].fillna('')
kpis = pd.DataFrame([
    {'Metrik': 'Cohort total',        'Wert': len(scores)},
    {'Metrik': 'Bots (score≥0.5)',    'Wert': int((scores['score'] >= 0.5).sum())},
    {'Metrik': 'Sofort-Follow',       'Wert': int(flags.str.contains('INSTANT_FOLLOW').sum())},
    {'Metrik': 'Anonyme Profile',     'Wert': int(flags.str.contains('ANONYMOUS_PROFILE').sum())},
    {'Metrik': 'Gelöschte Profile',   'Wert': int(flags.str.contains('DELETED').sum())},
    {'Metrik': 'Cluster-Knoten',      'Wert': len(result.cluster_nodes)},
    {'Metrik': 'Cluster-Kanten',      'Wert': len(result.cluster_edges)},
])
display(kpis)


## Top-Verdächtige

In [ ]:
top = scores.nlargest(20, 'score')[['handle', 'display_name', 'score', 'flags']]
display(top)


## Karten
Jede Karte ist eine eigenständige Plotly-Visualisierung im 1200×675-Format (Karte 5 ist 1200×1200).

In [ ]:
for card_id, fig in result.cards.items():
    print(card_id)
    fig.show()


## HTML-Dossier ins Lakehouse schreiben (optional)

In [ ]:
from pathlib import Path
from botfinder.dossier import render_dossier

try:
    out = Path('/lakehouse/default/Files/botfinder')
    out.mkdir(parents=True, exist_ok=True)
    figures = list(result.cards.values())
    flags = scores['flags'].fillna('')
    total = len(scores)
    stats = {
        'total_suspects':         total,
        'high_confidence_bots':   int((scores['score'] >= 0.7).sum()),
        'medium_confidence_bots': int(((scores['score'] >= 0.4) & (scores['score'] < 0.7)).sum()),
        'pct_instant_follow':     100 * flags.str.contains('INSTANT_FOLLOW').sum() / max(total, 1),
        'pct_anonymous':          100 * flags.str.contains('ANONYMOUS_PROFILE').sum() / max(total, 1),
        'mean_score':             float(scores['score'].mean()),
        'p90_score':              float(scores['score'].quantile(0.9)),
    }
    suspects = []
    for _, row in scores.nlargest(50, 'score').iterrows():
        suspects.append({
            'handle':          row.get('handle') or '',
            'display_name':    row.get('display_name') or '',
            'total_score':     float(row['score']),
            'flags':           [f for f in (row.get('flags') or '').split(',') if f],
            'posts_count':     0, 'followers_count': 0, 'age_minutes': -1,
            'source':          row.get('source') or '',
        })
    path = out / f'dossier_{config.anchor_slug}.html'
    render_dossier(config.anchor_handle, stats, figures, suspects, config.lookback_days, path)
    print(f'Dossier written to {path}')
except Exception as e:
    print(f'(Lakehouse write skipped: {e})')
